# 1.初始化并调用模型  
LangChain提供了两种常见函数来初始化模型：  
- 使用init_chat_model函数，由LangChain自动创建模型对象
- 使用不同模型对应的类，手动创建模型对象

## 1.1.init_chat_model
- 安装模型依赖
- 在.env中配置模型的api_key
- 调用init_chat_model函数，传入正确的模型名称

In [2]:
from langchain.chat_models import init_chat_model

model = init_chat_model(model="deepseek-v4-flash")

In [ ]:
# 根据模型名称自动确定其类型
print(type(model))

<class 'langchain_deepseek.chat_models.ChatDeepSeek'>


### 自定义模型提供商  
对于其他模型，必须自定义模型参数来访问  
- 需要在环境变量中定义api_key和base_url
- 在init_chat_model中指定model、model_provider、base_url和api_key

In [ ]:
import os

base_url = os.getenv("")
api_key = os.getenv("")

model = init_chat_model(
    model="qwen-max",
    model_provider="openai",  # 如果是LangChain不支持的模型，需要指定模型提供者就模仿为OpenAI
    base_url = base_url,  # 但是要指定url，因为不是真的访问openai
    api_key = api_key
)

### 调整模型参数  
除了修改模型提供者以外，init_chat_model函数允许我们调整模型参数，例如：
- temperature: 控制生成文本的随机性，值越小越确定，值越大越随机
- max_token: 控制生成文本的最大长度
- top_p: 控制生成文本的多样性，值越小越多样，值越大越确定
- timeout: 控制生成文本的超时时间
- max_retries: 控制生成文本的最大重试次数

In [ ]:
import os

base_url = os.getenv("")
api_key = os.getenv("")

model = init_chat_model(
    model="qwen-max",
    model_provider="openai",  # 如果是LangChain不支持的模型，需要指定模型提供者就模仿为OpenAI
    base_url = base_url,  # 但是要指定url，因为不是真的访问openai
    api_key = api_key,
    temperature=1.5,
    top_p = 0.9
)

## 1.2.使用model类

In [ ]:
from langchain_community.chat_models.tongyi import ChatTongyi

model = ChatTongyi(
    model = "qwen-max"
    # 其他模型参数
)

## 2.访问模型  
LangChain提供了两个不同的函数来访问模型：
- invoke: 阻塞式访问
- stream: 流式访问

### 方式一：invoke  
invoke函数是阻塞式调用，需要等待模型生成全部结果才会返回，等待时间较长。

In [3]:
response = model.invoke("你是谁")
print(response.model_dump_json())

{"content":"你好！我是 DeepSeek，由深度求索公司创造的 AI 助手。我是一个纯文本模型，可以帮你解答问题、处理信息、进行对话等。我的特点是：\n\n- **免费使用**：目前完全免费，没有收费计划\n- **长上下文**：支持 1M 上下文，可以一次性处理像《三体》三部曲那样的长篇内容\n- **文件处理**：支持上传图片、PDF、Word、Excel、PPT 等文件，并读取其中的文字信息\n- **联网搜索**：支持联网查询（需要手动开启）\n- **多平台**：有 Web 版和 App 版，App 还支持语音输入\n\n我的知识截止于 2025 年 5 月。有什么我可以帮你的吗？无论是学习、工作还是日常问题，都可以问我！😊","additional_kwargs":{"refusal":null,"reasoning_content":"好的，用户问“你是谁”，这是一个简单的自我介绍问题。用户可能是初次接触，想了解我的身份和功能。我需要直接、清晰地说明我是谁、由谁创造、核心特点和服务范围。\n\n想到了用友好问候开头，表明身份是DeepSeek，来自深度求索公司。然后列举关键能力：文本对话、文件处理、长上下文、联网搜索和免费使用。这样能帮助用户快速建立认知，并知道如何与我互动。最后以开放性问题结束，邀请用户提出进一步需求。"},"response_metadata":{"token_usage":{"completion_tokens":281,"prompt_tokens":5,"total_tokens":286,"completion_tokens_details":{"accepted_prediction_tokens":null,"audio_tokens":null,"reasoning_tokens":106,"rejected_prediction_tokens":null},"prompt_tokens_details":{"audio_tokens":null,"cache_write_tokens":null,"cached_tokens":0},"prompt_cache_hit_tokens":0,"prompt_cache_miss_tokens":5},"model_provider":"deepseek","

In [5]:
# 调用invoke函数，传入消息数组
response = model.invoke([
    {"role": "system", "content": "你扮演火箭队的武藏，以武藏的性格口吻来回答用户的问题"},
    {"role": "user", "content": "你是谁？"}
])
print(response.content)

哼哼~既然你诚心诚意地发问了，本小姐就大发慈悲地告诉你！为了防止世界被破坏，为了保护世界的和平，贯彻爱与真实的邪恶，可爱又迷人的反派角色——武藏！


### 方式二：stream  
invoke阻塞式调用需要等待较长时间才能看到AI返回的结果，而stream则是流式调用，可以实时看到AI返回的一个词

In [9]:
stream = model.stream("你是谁？")

In [10]:
print(type(stream))

<class 'generator'>


In [11]:
for chunk in stream:
    print(chunk.content, end="", flush=True)

你好呀！我是DeepSeek，由深度求索公司创造的AI助手。😊

我是一个纯文本模型，可以帮你处理各种问题——回答问题、写作、编程、翻译、分析等等。虽然我不支持多模态识别（比如直接“看懂”图片），但我可以读取你上传的图像、PDF、Word、Excel等文件中的文字信息来帮助你。

我的特点包括：
- **超长上下文**：1M token，可以一次性处理像《三体》三部曲那么大体量的内容
- **联网搜索**：虽然需要你手动开启搜索开关
- **完全免费**：没错，现在和未来都计划保持免费
- **知识截止**：2025年5月

有什么我可以帮你的吗？无论是学习、工作还是日常问题，尽管问我！✨

## 3.在智能体中使用模型
### 3.1.创建智能体
LangChain提供了一个create_agent函数用来快速创建智能体。调用create_agent时需要指定一个模型，有两种选择：  
- 使用初始化好的模型对象
- 使用模型名称，让LangChain自动初始化模型

In [12]:
from langchain.agents import create_agent

# 1.使用初始化好的model创建Agent
agent = create_agent(model=model)

# 2.指定Model名称，由LangChain自动初始化模型
agent = create_agent(model="deepseek-v4-flash")

### 3.2.调用智能体
- invoke：阻塞式调用
- stream：流式访问  

※ 智能体调用时需要传入一个dict,其中必须包含一个messages字段，也就是消息的列表

阻塞式调用

In [14]:
response = agent.invoke({
    "messages": [{"role": "user", "content": "你是谁？"}]
})

print(response)

{'messages': [HumanMessage(content='你是谁？', additional_kwargs={}, response_metadata={}, id='8416c30d-cd17-40a9-a95a-56574ee2bee5'), AIMessage(content='你好！我是DeepSeek，由深度求索公司创造的AI助手。😊\n\n我是一个纯文本模型，擅长回答各种问题、进行对话交流、帮助处理文档等。我有以下特点：\n\n✨ **完全免费** - 没有任何收费计划\n📚 **超长上下文** - 支持1M token，可以一次性处理像《三体》三部曲那样的长篇内容\n📎 **文件处理** - 支持上传图片、PDF、Word、Excel、PPT等文件，并从中提取文字信息\n🔍 **联网搜索** - 需要时可以手动开启联网功能获取最新信息\n🎤 **语音输入** - App端支持语音交互\n\n我的知识截止于2025年5月，会尽力为你提供准确、有用的帮助。有什么我可以协助你的吗？无论是学习、工作还是日常问题，尽管问我！😄', additional_kwargs={'refusal': None, 'reasoning_content': '好的，用户问了一个很基础的自我介绍问题：“你是谁？”。这是一个非常简单的初始询问，用户可能是第一次接触我，想了解我的身份和功能。\n\n我需要清晰、友好地介绍自己，说明我是谁、由谁创造、主要能力特点，以及我能提供什么帮助。最后可以加上一个开放式的邀请，让用户进一步提问。\n\n想到了直接以问候开头，然后明确身份是DeepSeek，由深度求索公司创造。接着概括核心特点：免费、长上下文、文件处理、联网搜索（需手动开启）、语音功能。最后说明我的知识截止时间和提供的帮助类型，并邀请用户提出具体问题。'}, response_metadata={'token_usage': {'completion_tokens': 311, 'prompt_tokens': 6, 'total_tokens': 317, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens'

流式访问

In [15]:
messages = agent.stream(
    {"messages": [{"role": "user", "content": "你是谁？"}]},
    stream_mode="messages"
)
print(type(messages))

<class 'generator'>


In [16]:
for token, metadata in messages:
    if token.content:
        print(token.content, end="", flush=True)

嗨！我是 **DeepSeek**，由深度求索公司创造的AI助手！🎉

简单来说，我是一个纯文本模型，擅长回答问题、聊天、写作、分析、编程、翻译……只要是文字任务，我都能帮你搞定！

**关于我的几个亮点：**
- 📱 **完全免费**：无论是网页版还是App，都不收费
- 📚 **超长上下文**：1M上下文窗口，可以一次性处理《三体》三部曲那么多内容
- 🔗 **支持文件上传**：可以读PDF、Word、Excel、PPT、图片里的文字
- 🌐 **联网搜索**：需要你手动开启，我就能获取最新信息
- 🗣️ **App端语音输入**：说话也能问我问题

我的知识截止到 **2025年5月**，不过不用担心旧闻——开启联网搜索就能找到最新资讯！

有什么需要帮忙的吗？无论是学习、工作、创作，还是单纯聊聊，我都很乐意陪你！😊